# スペクトル近似ノートブック

複数の基底スペクトルを線形結合して、ターゲットスペクトルを最もよく近似する重みを求めます。

| 手法 | 制約 | 特徴 |
|------|------|------|
| 非負最小二乗法（NNLS） | w ≥ 0 | 重みが0以上を保証 |
| 混合比制約 | w ≥ 0, Σw = 1 | 混合比として解釈できる |

**セルを上から順番に実行してください。**

## セル 1 ― ライブラリのインストール・インポート

In [ ]:
import subprocess, sys

try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
    import japanize_matplotlib

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import io
from scipy.optimize import nnls, minimize
from scipy.interpolate import interp1d

plt.rcParams['figure.dpi'] = 120
print('✅ 準備完了')

## セル 2 ― データの準備

**実データがある場合 → セル 2-B を実行**（セル 2-A はスキップ）  
**データがない場合 → セル 2-A を実行**（動作確認用サンプル）

### セル 2-A ― サンプルデータの生成（実データがない場合）

In [ ]:
np.random.seed(42)

def gaussian(x, mu, sigma, amp=1.0):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

wl = np.linspace(400, 800, 200)

_s1 = gaussian(wl, 480, 30) + gaussian(wl, 520, 20, 0.4); _s1 /= _s1.max()
_s2 = gaussian(wl, 600, 40) + gaussian(wl, 650, 15, 0.6); _s2 /= _s2.max()
_s3 = gaussian(wl, 700, 25) + gaussian(wl, 740, 20, 0.5); _s3 /= _s3.max()

_target = 0.5*_s1 + 0.3*_s2 + 0.2*_s3 + np.random.normal(0, 0.02, len(wl))
_target = np.clip(_target, 0, None)

basis_wls  = [wl, wl, wl]
basis_ints = [_s1, _s2, _s3]
target_wl  = wl
target     = _target

print('✅ サンプルデータ生成完了')
print(f'   基底スペクトル: {len(basis_wls)} 個（各 {len(wl)} 点）')
print(f'   ターゲット: {len(target_wl)} 点')

### セル 2-B ― 実データの読み込み（CSVファイル）

1. このセルを実行するとファイル選択ダイアログが開きます
2. 基底スペクトルのCSV + ターゲットのCSVをまとめて選択してください
3. **ファイル名のアルファベット順**で読み込まれ、最後のファイルがターゲットになります

CSVの形式（どちらでも自動判別）:  
- `1列目=波長, 2列目=強度`（ファイルを分けて管理）  
- `1列目=波長, 2列目=s1, 3列目=s2, ... 最終列=target`（1ファイルにまとめる）

In [ ]:
from google.colab import files

print('ファイルを選択してください...')
uploaded = files.upload()

def read_csv_robust(raw_bytes):
    """区切り文字・ヘッダー・空列を自動処理して数値配列を返す"""
    df = pd.read_csv(io.BytesIO(raw_bytes), header=None,
                     sep=None, engine='python', skip_blank_lines=True)
    df = df.dropna(axis=1, how='all')
    df = df[pd.to_numeric(df.iloc[:, 0], errors='coerce').notna()]
    df = df.apply(pd.to_numeric, errors='coerce').dropna()
    return df.values

filenames = sorted(uploaded.keys())
arrays = []
for fname in filenames:
    arr = read_csv_robust(uploaded[fname])
    arrays.append(arr)
    print(f'  {fname}: {arr.shape[0]} 行 × {arr.shape[1]} 列')

# パターン A: 複数ファイル（各ファイルに波長+強度）
if len(arrays) >= 2 and all(a.shape[1] >= 2 for a in arrays):
    basis_wls  = [a[:, 0] for a in arrays[:-1]]
    basis_ints = [a[:, 1] for a in arrays[:-1]]
    target_wl  = arrays[-1][:, 0]
    target     = arrays[-1][:, 1]
    print(f'\n✅ パターンA: 基底スペクトル {len(basis_wls)} 個 + ターゲット 1 個')

# パターン B: 1ファイルに複数列（1列目=波長, 以降=各スペクトル, 最終列=target）
elif len(arrays) == 1 and arrays[0].shape[1] >= 3:
    arr = arrays[0]
    wl_col    = arr[:, 0]
    n_spectra = arr.shape[1] - 1
    basis_wls  = [wl_col] * (n_spectra - 1)
    basis_ints = [arr[:, i+1] for i in range(n_spectra - 1)]
    target_wl  = wl_col
    target     = arr[:, -1]
    print(f'\n✅ パターンB: 基底スペクトル {len(basis_wls)} 個 + ターゲット 1 個')

else:
    raise ValueError('ファイルが1つだけ、または列数が不足しています。'
                     '基底スペクトル用CSVとターゲット用CSVの両方を選択してください。')

for i, (wl, sp) in enumerate(zip(basis_wls, basis_ints)):
    print(f'   s{i+1}: {len(wl)} 点  ({wl[0]:.1f} 〜 {wl[-1]:.1f})')
print(f'   target: {len(target_wl)} 点  ({target_wl[0]:.1f} 〜 {target_wl[-1]:.1f})')

## セル 3 ― 入力スペクトルの確認

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.tab10.colors

ax = axes[0]
for i, (wl, sp) in enumerate(zip(basis_wls, basis_ints)):
    ax.plot(wl, sp, label=f'基底 s{i+1}', color=colors[i])
ax.set_xlabel('波長')
ax.set_ylabel('強度')
ax.set_title(f'基底スペクトル（{len(basis_wls)} 個）')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(target_wl, target, color='black', lw=2, label='ターゲット')
ax.set_xlabel('波長')
ax.set_ylabel('強度')
ax.set_title('ターゲットスペクトル')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## セル 4 ― スペクトル近似

波長軸が異なる場合は共通グリッドに補間してから最小化します。

In [ ]:
# 共通波長グリッドに補間
all_wls   = basis_wls + [target_wl]
wl_common = np.linspace(max(wl.min() for wl in all_wls),
                        min(wl.max() for wl in all_wls), 500)

def interp(wl_src, intensity):
    return interp1d(wl_src, intensity, kind='linear',
                    bounds_error=False, fill_value=0.0)(wl_common)

s_interp   = [interp(wl, sp) for wl, sp in zip(basis_wls, basis_ints)]
target_fit = interp(target_wl, target)

A       = np.column_stack(s_interp)
n_basis = A.shape[1]

print(f'共通波長: {wl_common[0]:.1f} 〜 {wl_common[-1]:.1f}  ({len(wl_common)} 点)')
print(f'行列 A: {A.shape}  target: {target_fit.shape}')

# ── 診断: 基底スペクトル間の相関 ──────────────────────────
corr = np.corrcoef(A.T)
print('\n--- 基底スペクトルの相関行列 ---')
header = '       ' + '  '.join(f'  s{i+1} ' for i in range(n_basis))
print(header)
for i in range(n_basis):
    row = '  '.join(f'{corr[i,j]:+.3f}' for j in range(n_basis))
    print(f'  s{i+1}:  {row}')
print('（|値|が 0.9 以上 → 高相関 → 1成分に偏りやすい）')

# ── 手法 1: NNLS ─ w ≥ 0 ──────────────────────────────────
w_nnls, _   = nnls(A, target_fit)
fitted_nnls = A @ w_nnls
rmse_nnls   = np.sqrt(np.mean((fitted_nnls - target_fit) ** 2))

print('\n--- 手法1: 非負最小二乗法（NNLS） ---')
for i, w in enumerate(w_nnls):
    print(f'  w{i+1} = {w:.4f}')
print(f'  RMSE = {rmse_nnls:.6f}')

# ── 手法 2: 正則化 NNLS ─ 重みが1成分に偏るときに使う ──────
# λ を大きくするほど重みが均等化されるが RMSE は悪化する
lam = 0.01

A_reg = np.vstack([A,          lam * np.eye(n_basis)])
t_reg = np.concatenate([target_fit, np.zeros(n_basis)])

w_rnnls, _    = nnls(A_reg, t_reg)
fitted_rnnls  = A @ w_rnnls
rmse_rnnls    = np.sqrt(np.mean((fitted_rnnls - target_fit) ** 2))

print(f'\n--- 手法2: 正則化 NNLS (λ={lam}) ---')
for i, w in enumerate(w_rnnls):
    print(f'  w{i+1} = {w:.4f}')
print(f'  RMSE = {rmse_rnnls:.6f}')

# ── 手法 3: 混合比制約 ─ w ≥ 0 かつ Σw = 1 ──────────────
result = minimize(
    fun=lambda w: np.sum((A @ w - target_fit) ** 2),
    x0=np.ones(n_basis) / n_basis,
    method='SLSQP',
    bounds=[(0, None)] * n_basis,
    constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
)
w_mix      = result.x
fitted_mix = A @ w_mix
rmse_mix   = np.sqrt(np.mean((fitted_mix - target_fit) ** 2))

print('\n--- 手法3: 混合比制約（合計=1、非負） ---')
for i, w in enumerate(w_mix):
    print(f'  w{i+1} = {w:.4f}')
print(f'  合計 = {w_mix.sum():.6f}')
print(f'  RMSE = {rmse_mix:.6f}')

## セル 5 ― 近似結果の比較グラフ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, (title, fitted, weights, rmse) in zip(axes, [
    ('NNLS\n（非負）',
     fitted_nnls, w_nnls, rmse_nnls),
    (f'正則化 NNLS\n（λ={lam}、非負）',
     fitted_rnnls, w_rnnls, rmse_rnnls),
    ('混合比制約\n（合計=1、非負）',
     fitted_mix, w_mix, rmse_mix),
]):
    ax.plot(wl_common, target_fit, 'k-',  lw=2,   label='ターゲット', alpha=0.7)
    ax.plot(wl_common, fitted,     'r--', lw=1.8, label='近似結果')
    ax.fill_between(wl_common, target_fit, fitted,
                    alpha=0.15, color='red', label='残差')
    weight_str = '  '.join(f'w{i+1}={w:.3f}' for i, w in enumerate(weights))
    ax.set_title(f'{title}\n{weight_str}\nRMSE = {rmse:.4f}', fontsize=9)
    ax.set_xlabel('波長')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('強度')
plt.suptitle('スペクトル近似結果の比較', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('spectrum_approximation_result.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 spectrum_approximation_result.png として保存しました')

## セル 6 ― 重みの棒グラフ

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(n_basis)
w = 0.25

ax.bar(x - w,   w_nnls,  w, label='NNLS',       color='tab:blue')
ax.bar(x,       w_rnnls, w, label=f'正則化 NNLS (λ={lam})', color='tab:orange')
ax.bar(x + w,   w_mix,   w, label='混合比制約',   color='tab:green')

ax.set_xticks(x)
ax.set_xticklabels([f's{i+1}' for i in range(n_basis)])
ax.set_ylabel('重み')
ax.set_title('各手法で求めた重みの比較')
ax.legend()
ax.axhline(0, color='black', lw=0.8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## セル 7 ― 残差スペクトル

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wl_common, target_fit - fitted_nnls,
        label=f'NNLS  (RMSE={rmse_nnls:.4f})')
ax.plot(wl_common, target_fit - fitted_rnnls,
        label=f'正則化 NNLS  (RMSE={rmse_rnnls:.4f})')
ax.plot(wl_common, target_fit - fitted_mix,
        label=f'混合比制約 (RMSE={rmse_mix:.4f})')
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('波長')
ax.set_ylabel('残差（ターゲット − 近似）')
ax.set_title('残差スペクトル')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 参考 ― 手法の選び方

| 手法 | 特徴 | 向いている場面 |
|------|------|---------------|
| **NNLS** | 重みが 0 以上、制約なし | 基底スペクトルが十分に独立している場合 |
| **正則化 NNLS** | 重みが 0 以上、分散を促進 | 1成分に偏るとき。λ を調整して使う |
| **混合比制約** | 重みが 0 以上、合計 = 1 | 材料の混合比・成分分析 |

### 1成分に偏るときのチェックリスト

1. **相関行列を確認** → |値| ≥ 0.9 なら基底スペクトルが似すぎている
2. **正則化 NNLS を試す** → `lam` を 0.01 → 0.05 → 0.1 と増やす
3. **RMSE の変化を確認** → RMSE が大幅に悪化する場合、そのスペクトル成分はターゲットに実際に含まれていない可能性がある

### λ の目安

| λ | 挙動 |
|---|------|
| 0.001 | ほぼ通常 NNLS |
| 0.01 | バランス重視（最初に試す値） |
| 0.1 | 重みを強制的に分散。RMSE が悪化しやすい |